# `sparsegf2.circuits.config` - one validated circuit cell

`CircuitConfig` contains every non-sample knob for one circuit realization.
Construction validates the graph, picture, gate schedule, measurement rule,
depth, and simulator options before a run begins. Parameter sweeps and named
observables are already implemented in `sparsegf2.analysis`; they are not
future placeholders.

The six string graph families are `cycle`, `complete`, `path`, `lattice_2d`,
`newman_watts`, and `watts_strogatz`. A prebuilt `GraphTopology` or an arbitrary
simple undirected NetworkX graph can also be supplied through `from_networkx`.


## Current mode surface

- Pictures: `pure_state`, `purification`, `single_ref`.
- Gating: `brickwork`, `random_edge`, `random_pool`, `all_edges`.
- Matching for brickwork: `round_robin`, `palette`, `fresh`.
- Measurements: `bernoulli`, `gated`, `random_pair`, `uniform_count`.
- Depth: `O(n)`, `O(log_n)`, `until_purified`, or a literal
  `total_layers_override`.

`random_edge` samples distinct graph edges; `random_pool` samples with
replacement; `all_edges` fires the graph's stored edge list in deterministic
order. `uniform_count` chooses `meas_count` distinct candidates and then applies
the Bernoulli measurement probability to those candidates.


In [1]:
from sparsegf2.circuits import CircuitConfig
from sparsegf2.circuits.config import GATING_MODES
from sparsegf2.circuits.measurements import MEASUREMENT_MODES

print('gating modes     :', GATING_MODES)
print('measurement modes:', MEASUREMENT_MODES)
for mode in GATING_MODES:
    kwargs = {'gates_per_layer': 2} if mode == 'random_edge' else {}
    cfg = CircuitConfig(
        graph_spec='cycle', n=8, gating_mode=mode,
        total_layers_override=3, **kwargs,
    )
    print(f'{mode:>11}: layers={cfg.total_layers()}, expected ratio={cfg.expected_gate_to_meas_ratio():.3g}')


gating modes     : ('brickwork', 'random_edge', 'random_pool', 'all_edges')
measurement modes: ('bernoulli', 'gated', 'random_pair', 'uniform_count')
  brickwork: layers=3, expected ratio=3.33
random_edge: layers=3, expected ratio=1.67
random_pool: layers=3, expected ratio=3.33
  all_edges: layers=3, expected ratio=6.67


## Measurement modes and literal depth

The mode-specific fields are checked eagerly: `gates_per_layer` belongs only
to `random_edge` and `random_pool`, while `meas_count` belongs only to
`uniform_count`. A positive `total_layers_override` short-circuits both the
depth-mode formula and the random-edge gate-budget rescaling. This is the right
knob for an experiment specified at an exact measured depth.


In [2]:
for mode in MEASUREMENT_MODES:
    kwargs = {'meas_count': 3} if mode == 'uniform_count' else {}
    cfg = CircuitConfig(
        graph_spec='cycle', n=8, measurement_mode=mode, p=0.25,
        total_layers_override=5, **kwargs,
    )
    print(f'{mode:>13}: layers={cfg.total_layers()}, ratio={cfg.expected_gate_to_meas_ratio():.3g}')

scaled = CircuitConfig(
    graph_spec='cycle', n=8, gating_mode='random_edge',
    gates_per_layer=1, depth_factor=2,
)
literal = CircuitConfig(
    graph_spec='cycle', n=8, gating_mode='random_edge',
    gates_per_layer=1, depth_factor=2, total_layers_override=7,
)
print('formula / literal depth:', scaled.total_layers(), '/', literal.total_layers())


    bernoulli: layers=5, ratio=2
        gated: layers=5, ratio=2
  random_pair: layers=5, ratio=8
uniform_count: layers=5, ratio=5.33
formula / literal depth: 64 / 7


## Reproducibility and analysis

Schema v2 keys every sample by the pair `(base_seed, sample_seed)`, never by
their sum. Distinct pairs therefore cannot alias when a sweep changes both the
quenched graph seed and the trajectory seed. `simulate(..., analyses=...)`
computes registered or custom observables on the live final tableau, while
`sparsegf2.analysis.sweep` handles many configurations and seeds.


In [3]:
from sparsegf2.circuits import simulate

cfg = CircuitConfig(
    graph_spec='cycle', n=8, picture='purification', p=0.2,
    total_layers_override=4, base_seed=17,
)
record = simulate(
    cfg, sample_seed=3, analyses=['code_dimension', 'half_cut_entropy']
)
print('sample identity:', (cfg.base_seed, record.sample_seed))
print('online analyses:', record.analyses)
print('serialized depth override:', cfg.to_dict()['total_layers_override'])


sample identity: (17, 3)
online analyses: {'code_dimension': 3, 'half_cut_entropy': 2}
serialized depth override: 4


## Summary

`CircuitConfig` is the fail-fast description of one cell. It covers all six
named graph families, arbitrary NetworkX geometry, four gate schedules, four
measurement schedules, literal depth, schema-v2 pair seeding, and the current
online/offline analysis layer.
